In [1]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F
!pip install openai
import openai
import json

In [12]:
import json
from collections import defaultdict

match_id = 9045752693

with SparkSessionContext("case-3") as spark:

    data = spark.sql(f"""
        WITH events AS (
            SELECT
                CONCAT('kill:', CAST(id AS STRING)) AS event_id,
                'hero_kill' AS event_type,
                CAST(time AS INTEGER) AS event_time,
                CAST(steam_account_id AS BIGINT) AS steam_account_id,
                CAST(attacker AS INTEGER) AS attacker,
                CAST(target AS INTEGER) AS target,
                CAST(gold AS INTEGER) AS gold,
                CAST(xp AS INTEGER) AS xp,
                CAST(NULL AS INTEGER) AS cost,
                CAST(NULL AS INTEGER) AS radiant_mask,
                CAST(NULL AS INTEGER) AS dire_mask,
                CAST(NULL AS STRING) AS item_name,
                CAST(NULL AS INTEGER) AS item_cost,
                CAST(NULL AS INTEGER) AS streak_hero_id,
                CAST(NULL AS INTEGER) AS streak_value
            FROM lakehouse.raw.mongo__kill_events
            WHERE match_id = {match_id}
              AND id IS NOT NULL

            UNION ALL

            SELECT
                CONCAT('tower_destroyed:', CAST(id AS STRING)),
                'tower_destroyed',
                CAST(time AS INTEGER),
                CAST(NULL AS BIGINT),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(radiant AS INTEGER),
                CAST(dire AS INTEGER),
                CAST(NULL AS STRING),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER)
            FROM lakehouse.raw.mongo__tower_death_events
            WHERE match_id = {match_id}
              AND id IS NOT NULL

            UNION ALL

            SELECT
                CONCAT('roshan_reward:', CAST(id AS STRING)),
                'roshan_reward',
                CAST(time AS INTEGER),
                CAST(steam_account_id AS BIGINT),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(amount AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS STRING),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER)
            FROM lakehouse.raw.mongo__gold_events
            WHERE match_id = {match_id}
              AND id IS NOT NULL
              AND LOWER(reason) = 'roshan'

            UNION ALL

            SELECT
                CONCAT('buyback:', CAST(id AS STRING)),
                'buyback',
                CAST(time AS INTEGER),
                CAST(steam_account_id AS BIGINT),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(cost AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS STRING),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER)
            FROM lakehouse.raw.mongo__buyback_events
            WHERE match_id = {match_id}
              AND id IS NOT NULL

            UNION ALL

            SELECT
                CONCAT('purchase:', CAST(p.id AS STRING)),
                'item_purchase',
                CAST(p.time AS INTEGER),
                CAST(p.steam_account_id AS BIGINT),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                COALESCE(i.localized_name, i.name),
                CAST(i.cost AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER)
            FROM lakehouse.raw.mongo__purchase_events p
            INNER JOIN lakehouse.raw.pg__items i
                ON CAST(p.itemid AS INTEGER) = i.item_id
            WHERE p.match_id = {match_id}
              AND p.id IS NOT NULL
              AND i.cost > 2000

            UNION ALL

            SELECT
                CONCAT('streak:', CAST(s.id AS STRING)),
                LOWER(TRIM(s.type)),
                CAST(s.time AS INTEGER),
                CAST(s.steam_account_id AS BIGINT),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS STRING),
                CAST(NULL AS INTEGER),
                CAST(s.heroId AS INTEGER),
                CAST(s.value AS INTEGER)
            FROM lakehouse.raw.mongo__streak_events s
            WHERE s.match_id = {match_id}
              AND s.id IS NOT NULL
              AND UPPER(TRIM(s.type)) IN (
                  'KILL_STREAK',
                  'MULTI_KILL'
              )
        )

        SELECT *
        FROM events
        ORDER BY
            event_time ASC NULLS LAST,
            event_id ASC
    """)

    events = [
        row.asDict(recursive=True)
        for row in data.toLocalIterator()
    ]

    heroes = spark.sql("""
        SELECT DISTINCT
            hero_id,
            localized_name
        FROM lakehouse.raw.pg__heroes
        WHERE hero_id IS NOT NULL
    """).collect()

    players = spark.sql(f"""
        SELECT DISTINCT
            account_id,
            hero_id,
            is_radiant
        FROM lakehouse.raw.pg__match_players
        WHERE match_id = {match_id}
          AND hero_id IS NOT NULL
    """).collect()


if not events:
    raise ValueError(f"РњР°С‚С‡ {match_id} РЅРµ РЅР°Р№РґРµРЅ")


# hero_id -> РЅР°Р·РІР°РЅРёРµ РіРµСЂРѕСЏ
hero_names = {
    int(row.hero_id): row.localized_name
    for row in heroes
    if row.localized_name is not None
}


# account_id -> hero_id
account_heroes = defaultdict(set)

# hero_id -> РєРѕРјР°РЅРґР°
hero_sides = defaultdict(set)

for row in players:
    hero_id = int(row.hero_id)

    if row.account_id is not None:
        account_heroes[int(row.account_id)].add(hero_id)

    if row.is_radiant is not None:
        hero_sides[hero_id].add(bool(row.is_radiant))


# РЎРѕРїРѕСЃС‚Р°РІР»РµРЅРёРµ Р°РєРєР°СѓРЅС‚РѕРІ СЃ РіРµСЂРѕСЏРјРё
account_to_hero = {}

for account_id, hero_ids in account_heroes.items():
    if account_id in (0, 4294967295):
        continue

    if len(hero_ids) == 1:
        hero_id = next(iter(hero_ids))

        if hero_id in hero_names:
            account_to_hero[account_id] = hero_names[hero_id]


# True = Radiant, False = Dire
hero_teams = {}

for hero_id, sides in hero_sides.items():
    if len(sides) == 1:
        hero_teams[hero_id] = next(iter(sides))


def get_hero_name(hero_id):
    if hero_id is None:
        return None

    return hero_names.get(int(hero_id))


def get_account_hero(account_id):
    if account_id is None:
        return None

    return account_to_hero.get(int(account_id))


def get_kill_from_radiant(attacker_id, target_id):
    if attacker_id is None or target_id is None:
        return None

    attacker_team = hero_teams.get(int(attacker_id))
    target_team = hero_teams.get(int(target_id))

    if attacker_team is None or target_team is None:
        return None

    if attacker_team == target_team:
        return None

    return 1 if attacker_team else 0


# Р¤РѕСЂРјРёСЂСѓРµРј СЃРѕР±С‹С‚РёСЏ РґР»СЏ LLM
clean_events = []

for event in events:
    event_type = event["event_type"]

    fact = {
        "event_id": event["event_id"],
        "type": event_type,
        "time": event["event_time"]
    }

    if event_type == "hero_kill":
        fact["attacker"] = (
            get_hero_name(event["attacker"])
            or event["attacker"]
        )

        fact["target"] = (
            get_hero_name(event["target"])
            or event["target"]
        )

        fact["kill_from_radiant"] = get_kill_from_radiant(
            event["attacker"],
            event["target"]
        )

        fact["killer_account_id"] = event["steam_account_id"]
        fact["gold"] = event["gold"]
        fact["xp"] = event["xp"]

    elif event_type == "tower_destroyed":
        fact["radiant_mask"] = event["radiant_mask"]
        fact["dire_mask"] = event["dire_mask"]

    elif event_type == "roshan_reward":
        fact["hero"] = get_account_hero(
            event["steam_account_id"]
        )
        fact["account_id"] = event["steam_account_id"]
        fact["gold"] = event["gold"]

    elif event_type == "buyback":
        fact["hero"] = get_account_hero(
            event["steam_account_id"]
        )
        fact["account_id"] = event["steam_account_id"]
        fact["cost"] = event["cost"]

    elif event_type == "item_purchase":
        fact["hero"] = get_account_hero(
            event["steam_account_id"]
        )

        fact["account_id"] = event["steam_account_id"]
        fact["item"] = event["item_name"]
        fact["item_cost"] = event["item_cost"]

    elif event_type in ("kill_streak", "multi_kill"):
        # Р’ streak_events РЅР°РїСЂСЏРјСѓСЋ РїСЂРёСЃСѓС‚СЃС‚РІСѓРµС‚ heroId
        hero_id = event["streak_hero_id"]

        fact["hero"] = (
            get_hero_name(hero_id)
            or hero_id
        )

        fact["account_id"] = event["steam_account_id"]
        fact["value"] = event["streak_value"]

    # РЈРґР°Р»СЏРµРј NULL, РєСЂРѕРјРµ kill_from_radiant
    fact = {
        key: value
        for key, value in fact.items()
        if value is not None or key == "kill_from_radiant"
    }

    clean_events.append(fact)


# РЎРѕСЂС‚РёСЂСѓРµРј РІСЃРµ СЃРѕР±С‹С‚РёСЏ РїРѕ РёРіСЂРѕРІРѕРјСѓ РІСЂРµРјРµРЅРё
clean_events.sort(
    key=lambda e: (
        e.get("time") is None,
        e.get("time") if e.get("time") is not None else float("inf"),
        e["event_id"]
    )
)


match_payload = {
    "match_id": match_id,
    "events": clean_events
}

match_json = json.dumps(
    match_payload,
    ensure_ascii=False,
    separators=(",", ":")
)
print(
    json.dumps(
        match_payload,
        ensure_ascii=False,
        indent=2
    )
)

{
  "match_id": 9045752693,
  "events": [
    {
      "event_id": "kill:6af7fdde2652c8d2ed623041",
      "type": "hero_kill",
      "time": 15,
      "attacker": "Monkey King",
      "target": "Dragon Knight",
      "kill_from_radiant": 1,
      "killer_account_id": 107741011,
      "gold": 536,
      "xp": 66
    },
    {
      "event_id": "kill:6af7fdde2652c8d2ed62304d",
      "type": "hero_kill",
      "time": 218,
      "attacker": "Warlock",
      "target": "Queen of Pain",
      "kill_from_radiant": 1,
      "killer_account_id": 177076440,
      "gold": 314,
      "xp": 269
    },
    {
      "event_id": "kill:6af7fdde2652c8d2ed623068",
      "type": "hero_kill",
      "time": 233,
      "attacker": "Rubick",
      "target": "Shadow Fiend",
      "kill_from_radiant": 0,
      "killer_account_id": 341106989,
      "gold": 346,
      "xp": 880
    },
    {
      "event_id": "kill:6af7fdde2652c8d2ed623054",
      "type": "hero_kill",
      "time": 259,
      "attacker": "Axe",
     

In [13]:
import openai
import os
import json

SYSTEM_PROMPT = """
Ты описываешь события матча Dota 2 на русском языке.

На входе JSON одного матча с match_id и массивом events.
События расположены по игровому времени. У каждого события есть
event_id, type, time (целые секунды) и поля с данными.

Для каждого события верни один объект с полями event_id и description.
В event_id точно скопируй event_id исходного события. В description напиши одно
короткое предложение, которое объясняет, что произошло в игре.
Для события с итоговой статистикой опиши соответствующий результат.

Сохрани порядок массива events. Количество объектов в ответе должно
равняться количеству событий на входе.

Используй только данные текущего события. Если есть игровое время,
укажи его в формате М:СС. Если часть данных отсутствует, опиши то,
что известно. Сохраняй указанный в событии статус подтверждения.
attacker и target — имена героев: сохраняй их написание.
killer_account_id — идентификатор аккаунта, а не имя героя.
Для hero_kill укажи, кто кого убил; gold и xp упоминай только при наличии.
Для tower_destroyed сообщи о разрушении башни. Не определяй её сторону,
линию или уровень по radiant_mask/dire_mask без явной расшифровки.
Не называй первое по времени убийство первой кровью без явного подтверждения.
Не объявляй победителя или окончание матча, если таких данных нет.

Верни только JSON, без Markdown и пояснений. Формат ответа:
{"events": [{"event_id": "kill:example", "description": "На 0:26 Lone Druid убил Outworld Devourer."}]}

Если events пуст, верни {"events": []}.
"""

match_json = json.dumps(match_payload, ensure_ascii=False)


def parse_events_response(content):
    if not isinstance(content, str) or not content.strip():
        raise ValueError("Первый ответ модели пуст.")
    text = content.strip()
    lines = text.splitlines()
    if len(lines) >= 3 and lines[0].lower() in ("```json", "```") and lines[-1] == "```":
        text = "\n".join(lines[1:-1])
    try:
        payload = json.loads(text)
    except json.JSONDecodeError as exc:
        raise ValueError("Первый ответ модели не является корректным JSON.") from exc
    if not isinstance(payload, dict) or not isinstance(payload.get("events"), list):
        raise ValueError("Первый ответ должен содержать массив events.")
    return payload["events"]

client = openai.OpenAI(
    api_key="sk-67qOmCQbXkp39cHsUJboMw",
    base_url="https://api.vsellm.ru/v1"
)

response = client.chat.completions.create(
    model="openai/gpt-5.4-nano",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": match_json}
    ]
)

print(response.choices[0].message.content)

{"events":[{"event_id":"kill:6af7fdde2652c8d2ed623041","description":"На 0:15 Monkey King убил Dragon Knight, получив 536 золота и 66 опыта."},{"event_id":"kill:6af7fdde2652c8d2ed62304d","description":"На 3:38 Warlock убил Queen of Pain, получив 314 золота и 269 опыта."},{"event_id":"kill:6af7fdde2652c8d2ed623068","description":"На 3:53 Rubick убил Shadow Fiend, получив 346 золота и 880 опыта."},{"event_id":"kill:6af7fdde2652c8d2ed623054","description":"На 4:19 Axe убил Ogre Magi, получив 314 золота и 269 опыта."},{"event_id":"kill:6af7fdde2652c8d2ed62305e","description":"На 5:11 Queen of Pain убила Axe."},{"event_id":"kill:6af7fdde2652c8d2ed623045","description":"На 5:16 Witch Doctor убил Dragon Knight, получив 330 золота и 722 опыта."},{"event_id":"kill:6af7fdde2652c8d2ed623046","description":"На 5:18 Witch Doctor убил Ogre Magi, получив 330 золота и 688 опыта."},{"event_id":"streak:6af7fdde2652c8d2ed625c6a","description":"На 5:18 Witch Doctor оформил мультикилл на 2."},{"event_id":"

In [16]:
import openai
import os
import json

SUMMARY_PROMPT = """
Напиши короткий послематчевый обзор Dota 2 на русском языке. Пиши спокойно и естественно. Покажи, чем запомнился каждый отрезок игры, и объединяй события в связный рассказ.

На входе JSON одного матча. В массиве events события уже идут по игровому времени. У каждого есть event_id, type, time и данные события. Поле description может содержать готовое описание. Поле time указано в секундах или равно null. Используй структурированные поля для команд, героев, предметов, чисел и таймкодов. Если поля и описание противоречат друг другу, опирайся на поля.

Структура обзора

- 0 ≤ time < 600 — «На стадии лейнинга…»
- 600 ≤ time < 1500 — «В ранней игре…»
- 1500 ≤ time < 2400 — «На средней стадии игры…»
- 2400 ≤ time ≤ 3900 — «В поздней игре…»
- time > 3900 — «В гиперлейте…»

На каждую непустую стадию дай один абзац из 2–4 предложений. Если на стадии есть только один значимый факт, достаточно одного предложения. Пропускай пустые стадии. Соблюдай хронологию внутри абзаца. Если первое подтверждённое игровое событие имеет type="first_blood", начни с «Игра началась с…» и упомяни первую кровь один раз. Самый ранний hero_kill сам по себе первую кровь не подтверждает. Важные события без таймкода опиши отдельно перед итогом. Предматчевые события при необходимости упомяни перед лейнингом.

Какие события выделять

- В hero_kill значение kill_from_radiant=1 означает убийство Radiant, значение 0 — убийство Dire. По учтённым убийствам можно показать, у кого был перевес на стадии. Если команда неизвестна, не учитывай это убийство в сравнении. По убийствам можно судить о перевесе по убийствам, а исход матча бери из подтверждённого итога.
- Показывай подтверждённые серии от двух убийств одного героя или игрока. MULTI_KILL описывай как несколько быстрых убийств, KILL_STREAK — как серию убийств без смерти. Бери размер из события, если он указан. Если отдельного события серии нет, объедини близкие hero_kill одного героя и назови число убийств, которое видно во входе. Двойное или тройное убийство называй так только при подтверждённом MULTI_KILL. Если серия и записи об убийствах описывают один эпизод, упомяни его один раз.
- Из покупок выбирай заметные законченные предметы с известным владельцем. Укажи героя, предмет и таймкод, если он есть. Например, «На 13:40 Axe купил Blink Dagger». Заявляй об использовании предмета или его влиянии на события только при прямом подтверждении во входе.
- Близкие падения башен можно объединить в один эпизод. Сторону, линию и тир указывай, если они известны. Объект tower_destroyed считай концом матча, если его данные явно подтверждают, что пал трон. После подтверждённого конца игровые стадии заканчиваются. Победителя, счёт и длительность бери из match_result или других явных данных об итоге. Если итог известен, вынеси его в последний абзац. В остальных случаях закончи последним подтверждённым эпизодом.

Указывай таймкоды ключевых эпизодов в формате М:СС. Границы стадий, сырые маски, золото и опыт за отдельные убийства опускай. Сохраняй имена, команды и подтверждённые числа. Используй только события из входных данных и явно подтверждённые связи между ними. Верни только текст обзора.
"""

first_answer = response.choices[0].message.content
first_events = parse_events_response(first_answer)
source_events = match_payload["events"]

summary_payload = {
    "match_id": match_payload["match_id"],
    "events": [
        {
            "event_id": source["event_id"],
            "type": source["type"],
            "time": source.get("time"),
            "description": described["description"],
        }
        for source, described in zip(source_events, first_events)
    ],
}

summary_response = client.chat.completions.create(
    model="openai/gpt-5.4-nano",
    messages=[
        {"role": "system", "content": SUMMARY_PROMPT},
        {"role": "user", "content": json.dumps(summary_payload, ensure_ascii=False)},
    ],
)

print(summary_response.choices[0].message.content)

Игра началась с быстрых разменов: на 0:15 Monkey King убил Dragon Knight, а затем к 3:53 разогнались сайд-скирмиши с убийствами Queen of Pain, Shadow Fiend и Ogre Magi. Линии постоянно перетряхивали составы, и ближе к 5-й минуте темп удерживала Witch Doctor, оформив мультикилл на 2 (5:18). На 5:16 Queen of Pain также закрыла Axe, после чего стороны обменялись ещё несколькими точечными убийствами.

В ранней игре началась серия перестрелок в окрестностях мидов и лесных стыков: Witch Doctor продолжал выигрывать моменты и на 6:59 оформил серию на 3. Параллельно Rubick активно отвечал по карте — сначала поймал Shadow Fiend, а затем разогнал собственные убийства до длинной серии, поддерживая контроль за темпом. На 7:15 и 7:47 башни уже начали падать, а к 8:23 Witch Doctor закрепила преимущество в небольшом отрезке, забрав Queen of Pain. Из покупок запомнились точечные ускорения: Phylactery у Rubick (5:37), Blink Dagger у Axe (7:01), Diffusal Blade у Monkey King (6:51) и Boots of Travel у нес